In [1]:
# Initialize Otter
import otter
from numpy import ndarray

grader = otter.Notebook("DS3000-9000_A05.ipynb")

# Assignment 5: Bias-Variance Tradeoff

**DS 3000A/9000A Fall 2026**

This assignment will be graded on Gradescope.

1. Run `pip install otter-grader` first, if you don't have it installed.
2. Ellipsis (`...`) in a Python code cell is a placeholder. You need to replace it with your solution.
3. Included in this notebook are public test cases, which you can use to verify your answers before submitting.
4. Additional private test cases will run on Gradescope upon submission. You will see your score but not the tests themselves.
5. You can revise and re-upload your submission on Gradescope up until the deadline.

In [2]:
## Install packages if neccessary.
## Uncomment and run if you need to.

# !pip install numpy scipy pandas scikit-learn

In [3]:
import numpy as np

---

## Question 1
 
**50 Points**
 
In the lecture we have seen the theoretical decomposition of the MSE of an estimator $f_{\mathcal{D}}(\mathbf{x})$ at a test point $\mathbf{x}$ into bias and variance:

$$
    \mathrm{MSE}(\mathbf{x}) = \mathrm{Bias}^2(\mathbf{x}) + \mathrm{Var}(\mathbf{x}) ,
$$

where:

- $\mathrm{MSE}(\mathbf{x}) = \mathbb{E}_{\mathcal{D} \sim p(\mathcal{D})}[(f^*(\mathbf{x}) - f_{\mathcal{D}}(\mathbf{x}))^2]$,

- $\mathrm{Bias}(\mathbf{x}) = \mathbb{E}_{\mathcal{D} \sim p(\mathcal{D})}[f_{\mathcal{D}}(\mathbf{x}) - f^*(\mathbf{x})]$,

- $\mathrm{Var}(\mathbf{x}) = \mathbb{E}_{\mathcal{D} \sim p(\mathcal{D})}[(f_{\mathcal{D}}(\mathbf{x}) - \mathbb{E}_{\mathcal{D} \sim p(\mathcal{D})}[f_{\mathcal{D}}(\mathbf{x})])^2]$.

Here, $p(\mathcal{D})$ is the distribution over training datasets, $f^*(\mathbf{x})$ is the true function value at $\mathbf{x}$, and $f_{\mathcal{D}}(\mathbf{x})$ is the estimator trained on dataset $\mathcal{D}$.

Suppose we have a 1D regression problem:

- $f^*(x) = \sin(2 \pi x)$,

- $p(\mathcal{D})$ is the distribution over datasets of size $N=10$ where each dataset consists of $N$ i.i.d. samples $(x_i, y_i)$ with $x_i \sim \mathrm{Uniform}(0, 1)$ and $y_i = f^*(x_i) + \epsilon_i$, where $\epsilon_i \sim \mathcal{N}(0, 0.1^2)$.

- $f_{\mathcal{D}}(x)$ is the estimator obtained by fitting a polynomial of degree $d$ to the dataset $\mathcal{D}$ using least squares regression.


In [4]:
np.random.seed(30009000)


def estimator(
    x_test: float, x_train: np.ndarray, y_train: np.ndarray, degree: int
) -> float:
    """Fit a polynomial of given degree to a random dataset and predict at x_test.

    Args:
        x_test: Test point to predict at.
        x_train: Training input data, shape (N,).
        y_train: Training target data, shape (N,).
        degree: Degree of the polynomial to fit.

    Returns:
        Predicted value at x_test as a float.
    """
    coeffs = np.polyfit(x_train, y_train, degree)
    return np.polyval(coeffs, x_test)

1. Implement the function to sample a random dataset $\mathcal{D}$ from $p(\mathcal{D})$ as described above.

2. Implement the function compute $\mathrm{Bias}(x)$. Use $M=1000$ random datasets to estimate the expectation over $\mathcal{D}$.

3. Similarly, implement the function to compute $\mathrm{Var}(x)$ using $M=1000$ random datasets.

4. Implement the function to compute $\mathrm{MSE}(x)$ using $M=1000$ random datasets.

5. Verify that the bias-variance decomposition holds by checking that $\mathrm{MSE}(x) \approx \mathrm{Bias}^2(x) + \mathrm{Var}(x)$ for a test point $x=0.5$ and polynomial degree $3$. Store the bias, variance, MSE, and the decomposition error $\lvert \mathrm{MSE}(0.5) - (\mathrm{Bias}(0.5)^2 + \mathrm{Var}(0.5)) \rvert$ in variables `bias_x`, `var_x`, `mse_x`, and `decomposition_error` respectively, and print the decomposition error.

**HINT:** Use Monte Carlo estimation to compute the expectations over $\mathcal{D}$ by sampling multiple random datasets and averaging the results.

In [5]:
def sample_dataset(N: int) -> tuple[np.ndarray, np.ndarray]:
    """Sample a random dataset from p(D).

    Args:
        N: Number of samples in the dataset.

    Returns:
        A tuple (x_train, y_train) where x_train and y_train are arrays of shape (N,).
    """
    x_train = np.random.random_sample(N)
    y_train = np.sin(2*np.pi*x_train) + np.random.normal(0, 0.1, N)
    return (x_train, y_train)

    
def bias(x_test: float, degree: int, M: int = 1000) -> float:
    """Compute the bias at x_test.

    Args:
        x_test: Test point to compute bias at.
        degree: Degree of the polynomial to fit.
        M: Number of random datasets to sample for Monte Carlo estimation.

    Returns:
        Bias at x_test as a float.
    """
    trueVal = np.sin(2*x_test*np.pi)
    res = np.zeros(M)
    for i in range(M):
        x_train, y_train = sample_dataset(N=10)
        res[i] = estimator(x_test, x_train, y_train, degree)
    return np.average(res)

def variance(x_test: float, degree: int, M: int = 1000) -> float:
    """Compute the variance at x_test.

    Args:
        x_test: Test point to compute variance at.
        degree: Degree of the polynomial to fit.
        M: Number of random datasets to sample for Monte Carlo estimation.

    Returns:
        Variance at x_test as a float.
    """
    estimates = np.zeros(M)
    for i in range(M):
        x_train, y_train = sample_dataset(N=10)
        estimates[i] = estimator(x_test, x_train, y_train, degree)
    return np.average((estimates - np.average(estimates))**2)


def mse(x_test: float, degree: int, M: int = 1000) -> float:
    """Compute the MSE at x_test.

    Args:
        x_test: Test point to compute MSE at.
        degree: Degree of the polynomial to fit.
        M: Number of random datasets to sample for Monte Carlo estimation.

    Returns:
        MSE at x_test as a float.
    """
    B = bias(x_test, degree, M)
    V = variance(x_test, degree, M)
    return B**2 + V

x_test = 0.5
degree = 3

bias_x = bias(x_test, degree, M=1000)
var_x = variance(x_test, degree, M=1000)
mse_x = mse(x_test, degree, M=1000)
decomposition_error = np.abs(mse_x - (bias_x**2 + var_x))

print(
    f"Decomposition error at x = {x_test} is approximately: {decomposition_error:.6f}"
)

Decomposition error at x = 0.5 is approximately: 0.000246


In [6]:
grader.check("q1")

q1 results: All test cases passed!

---

## Question 2
 
**50 Points**
 
A model's test MSE is only a trustworthy estimate of generalization if the test set was never touched during training or model selection. Violating this produces **data leakage** --- the reported test MSE looks better than it actually is.
 
We use a synthetic regression dataset and polynomial regression via `sklearn`:

In [7]:
from sklearn.datasets import make_regression
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures

np.random.seed(30009000)
X, y = make_regression(n_samples=100, n_features=1, noise=30, random_state=30009000)

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=30009000
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=30009000
)

degrees = list(range(1, 10))

This gives a 60-20-20 train/val/test split.
 
Implement the three pipelines below. For each, select the best polynomial degree, then report the test MSE of the selected model.
 
- **Pipeline A (correct):** Select degree using $\mathcal{D}_\text{val}$, train final model on $\mathcal{D}_\text{train}$, evaluate on $\mathcal{D}_\text{test}$.

- **Pipeline B (test leakage):** Select degree by evaluating directly on $\mathcal{D}_\text{test}$, train final model on $\mathcal{D}_\text{train}$, evaluate on $\mathcal{D}_\text{test}$.

- **Pipeline C (train leakage):** Select degree using $\mathcal{D}_\text{val}$, train final model on $\mathcal{D}_\text{train} \cup \mathcal{D}_\text{test}$, evaluate on $\mathcal{D}_\text{test}$.

**The key question:** Do pipelines B and C report lower test MSE than pipeline A? 
If so, by how much? 
A lower test MSE sounds like a better model, but it isn't. 
It's an artifact of the leakage. 
The model hasn't generalized better; you've just broken the rules of evaluation.
This is a very common user error in the real-world. **Watch out!**
 
**Assign:**
- `best_degree_a`, `test_mse_a` for Pipeline A

- `best_degree_b`, `test_mse_b` for Pipeline B

- `best_degree_c`, `test_mse_c` for Pipeline C
 
**Hint:** Use `make_pipeline(PolynomialFeatures(degree=d), Ridge())` to fit a degree-$d$ polynomial regression model.

In [8]:
def pipeline_a(
    X_train, y_train, X_val, y_val, X_test, y_test, degrees
) -> tuple[int, float]:
    """Correct pipeline: select degree on val, evaluate on test.

    Returns:
        Tuple (best_degree, test_mse).
    """
    best_degree, best_mse = 0, np.inf
    for degree in degrees:
        poly = make_pipeline(PolynomialFeatures(degree=degree), Ridge())
        poly.fit(X_val, y_val)
        mse = mean_squared_error(poly.predict(X_val), y_val)
        if mse < best_mse:
            best_degree = degree
            best_mse = mse
    poly = make_pipeline(PolynomialFeatures(degree=best_degree), Ridge())
    poly.fit(X_train, y_train)
    return best_degree, mean_squared_error(poly.predict(X_test), y_test)
    

def pipeline_b(X_train, y_train, X_test, y_test, degrees) -> tuple[int, float]:
    """Test leakage: select degree on test, evaluate on test.

    Returns:
        Tuple (best_degree, test_mse).
    """
    best_degree, best_mse = 0, np.inf
    for degree in degrees:
        poly = make_pipeline(PolynomialFeatures(degree=degree), Ridge())
        poly.fit(X_test, y_test)
        mse = mean_squared_error(poly.predict(X_test), y_test)
        if mse < best_mse:
            best_degree = degree
            best_mse = mse
    poly = make_pipeline(PolynomialFeatures(degree=best_degree), Ridge())
    poly.fit(X_train, y_train)
    return best_degree, mean_squared_error(poly.predict(X_test), y_test)


def pipeline_c(
    X_train, y_train, X_val, y_val, X_test, y_test, degrees
) -> tuple[int, float]:
    """Train leakage: select degree on val, train final model on train+test.

    Returns:
        Tuple (best_degree, test_mse).
    """
    best_degree, best_mse = 0, np.inf
    for degree in degrees:
        poly = make_pipeline(PolynomialFeatures(degree=degree), Ridge())
        poly.fit(X_val, y_val)
        mse = mean_squared_error(poly.predict(X_val), y_val)
        if mse < best_mse:
            best_degree = degree
            best_mse = mse
    poly = make_pipeline(PolynomialFeatures(degree=best_degree), Ridge())
    X_train_and_test = np.append(X_train, X_test, axis=0)
    Y_train_and_test = np.append(y_train, y_test)
    poly.fit(X_train_and_test, Y_train_and_test)
    return best_degree, mean_squared_error(poly.predict(X_test), y_test)


best_degree_a, test_mse_a = pipeline_a(X_train, y_train, X_val, y_val, X_test, y_test, degrees)
best_degree_b, test_mse_b = pipeline_b(X_train, y_train, X_test, y_test, degrees)
best_degree_c, test_mse_c = pipeline_c(X_train, y_train, X_val, y_val, X_test, y_test, degrees)

print(f"Pipeline A (correct)       degree={best_degree_a}, test MSE={test_mse_a:.2f}")
print(f"Pipeline B (test leakage)  degree={best_degree_b}, test MSE={test_mse_b:.2f}")
print(f"Pipeline C (train leakage) degree={best_degree_c}, test MSE={test_mse_c:.2f}")

Pipeline A (correct)       degree=9, test MSE=563.35
Pipeline B (test leakage)  degree=9, test MSE=563.35
Pipeline C (train leakage) degree=9, test MSE=500.15


In [9]:
grader.check("q2")

q2 results: All test cases passed!

## Submission

Make sure you have run all cells in your notebook in order before running the cell below, so that all images/graphs appear in the output. The cell below will generate a zip file for you to submit. **Please save before exporting!**

In [10]:
# Save your notebook first, then run this cell to export your submission.
grader.export(pdf=False, run_tests=True)

Running your submission against local test cases...


Your submission received the following results when run against available test cases:

    q1 results: All test cases passed!

    q2 results: All test cases passed!
